## Business Understanding

## Imports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Understanding

In [45]:
df = pd.read_csv('../data/judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [46]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [48]:
df.shape

(9093, 3)

## Data Preparation
1. **Lemmatization**- examines the morphology of the words and reduces each word to its most basic form or lemma eg studies become study.

2. **Lowercasing**- reduces the number of unique words words the model must handle, improves model consistency and ensures the model learns **all occurences of a word together**, and simplifies **text processing**

3. **Tokenization and removing stopwords**-splits the sentences into array of individual words. Stopwords(common words or pretty useless data) that contain litte or no information.

4. **Label Encoding**-a simple and very common technique in Machine Learning to convert **categorical labels**(text) into **numbers**.

5. **Stemming**- stemming refers to the removal of suffixes eg cats to cat.

In [49]:
df.isnull().sum() ## check for null entries



tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [50]:
# dealing with the missing values. Do we drop of fill?
# for tweet, we will drop that one missing value

df = df.dropna(subset=['tweet_text']) # dropped the one missing tweet
df.isnull().sum() 

tweet_text                                               0
emotion_in_tweet_is_directed_at                       5801
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [51]:
# Duplicates
duplicates = df['tweet_text'].duplicated().sum()  # check for duplicates in the 'tweet' column
print(f"Duplicate tweets: {duplicates}")
df = df.drop_duplicates(subset='tweet_text').reset_index(drop=True) # drop duplicates based on the 'tweet' column and reset index
print(f"Shape after removing duplicates: {df.shape}")

Duplicate tweets: 27
Shape after removing duplicates: (9065, 3)


## Text Cleaning Function

In [52]:
## Mapping the product names to their respective brands

apple_list = [x.lower() for x in ['iPad', 'Apple', 'iPad app', 'iPhone app', 'iPhone', 'Mac', 'MacBook']]
google_list = [x.lower() for x in ['Google' , 'Android App', 'Android']]

def map_to_brand(product_name):
    if pd.isna(product_name):
        return None
    
    product_name = str(product_name).lower()
    
    if product_name in apple_list:
        return 'Apple'
    elif product_name in google_list:
        return 'Google'
    else:
        return None

df['product'] = df['emotion_in_tweet_is_directed_at'].apply(map_to_brand)

def fill_missing_product(row):
    if pd.notna(row['product']):
        return row['product']
    
    text = str(row['tweet_text']).lower()
    
    if any(word in text for word in ['apple', 'ipad', 'iphone', 'mac']):
        return 'Apple'
    elif any(word in text for word in ['google', 'android', 'pixel']):
        return 'Google'
    else:
        return 'Not specified'

df['product'] = df.apply(fill_missing_product, axis=1)

In [53]:
stop_words = set(stopwords.words('english'))

def get_clean_tokens(text):
    if not isinstance(text, str):
        return []
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtag symbols (keep the word)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stop words
    
    tokens = [word for word in tokens if word not in stop_words]
    # Remove short tokens (1-2 chars)
    tokens = [t for t in tokens if len(t) > 2]
    return tokens

In [54]:
## Apply the cleaning function to the 'tweet_text' column and create a new column 'clean_text'

df['clean_text'] = df['tweet_text'].apply(lambda x: " ".join(get_clean_tokens(x)))

df[['tweet_text', 'clean_text']].head()

,tweet_text,clean_text
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iphone hrs tweeting riseaustin dead need upgra...
1,@jessedee Know about @fludapp ? Awesome iPad/i...,know awesome ipadiphone app youll likely appre...
2,@swonderlin Can not wait for #iPad 2 also. The...,wait ipad also sale sxsw
3,@sxsw I hope this year's festival isn't as cra...,hope years festival isnt crashy years iphone a...
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,great stuff fri sxsw marissa mayer google tim ...


In [58]:
positive_emotion = []
negative_emotion = []


for text, is_there_an_emotion_directed_at_a_brand_or_product in zip(df['tweet_text'], df['is_there_an_emotion_directed_at_a_brand_or_product']):
    tokens = get_clean_tokens(text)
    if is_there_an_emotion_directed_at_a_brand_or_product == 'Positive emotion':
        positive_emotion.extend(tokens)

    elif is_there_an_emotion_directed_at_a_brand_or_product == 'Negative emotion':
        negative_emotion.extend(tokens)

# Top 15 Positive tweets
print("\n=== Top 15 Words in POSITIVE Sentiment ===")
print(Counter(positive_emotion).most_common(15))

# Top 15 Negative tweets
print("\n=== Top 15 Words in NEGATIVE Sentiment ===")
print(Counter(negative_emotion).most_common(15))


=== Top 15 Words in POSITIVE Sentiment ===
[('sxsw', 3102), ('link', 1206), ('ipad', 1191), ('apple', 835), ('google', 634), ('store', 538), ('iphone', 522), ('app', 394), ('new', 359), ('austin', 288), ('popup', 218), ('android', 196), ('amp', 178), ('launch', 160), ('get', 158)]

=== Top 15 Words in NEGATIVE Sentiment ===
[('sxsw', 578), ('ipad', 194), ('iphone', 155), ('google', 140), ('link', 101), ('apple', 99), ('app', 60), ('store', 44), ('new', 43), ('like', 39), ('circles', 33), ('social', 30), ('apps', 30), ('people', 29), ('design', 28)]


## TF -IDF

TF - Maps out the frequency of tokens in our data
IDF - shows the relevance/Weight of words to our target variable

In [60]:
## checking the total vocabulary size
vectorizer = TfidfVectorizer()
vectorizer.fit(df['clean_text'])
vocab_size = len(vectorizer.vocabulary_)
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 9968


In [63]:
tfidf = TfidfVectorizer(max_features=10000)

X = tfidf.fit_transform(df['clean_text'])
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

In [64]:
print(X)

  (0, 8539)	0.04559480881152363
  (0, 8271)	0.40660293552979493
  (0, 6374)	0.3835598129500589
  (0, 9325)	0.3371141271098152
  (0, 5612)	0.21876318548099558
  (0, 2097)	0.3257774421286677
  (0, 7403)	0.40660293552979493
  (0, 9184)	0.3091895366090419
  (0, 4045)	0.375335366511452
  (0, 4403)	0.12665731015129197
  (1, 3260)	0.2006334002968576
  (1, 3489)	0.27612589144000754
  (1, 8844)	0.29836520177139864
  (1, 222)	0.2729490398194457
  (1, 2194)	0.24890120427620757
  (1, 401)	0.3937050050691054
  (1, 4883)	0.35361314430940316
  (1, 9916)	0.308923755742757
  (1, 355)	0.16449107305764885
  (1, 4382)	0.369534107515847
  (1, 610)	0.24816758777975104
  (1, 4678)	0.23464632664403373
  (1, 8539)	0.0478263069134949
  (2, 7510)	0.6167572790747662
  (2, 4371)	0.2049381676706273
  :	:
  (9062, 5647)	0.20092042637488827
  (9062, 6465)	0.23472981684484198
  (9062, 9903)	0.1851999550547468
  (9062, 3573)	0.15054737119768363
  (9062, 8539)	0.031105675100614082
  (9063, 9911)	0.37876520797125046
  (9

In [65]:
print(y)

0                         Negative emotion
1                         Positive emotion
2                         Positive emotion
3                         Negative emotion
4                         Positive emotion
                       ...                
9060                      Positive emotion
9061    No emotion toward brand or product
9062    No emotion toward brand or product
9063    No emotion toward brand or product
9064    No emotion toward brand or product
Name: is_there_an_emotion_directed_at_a_brand_or_product, Length: 9065, dtype: object


## Train/Test Split

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Modelling

### Logistic Regresison

In [68]:
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

In [69]:
print(classification_report(y_test, y_pred_log))

                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       0.36      0.64      0.46       114
No emotion toward brand or product       0.77      0.66      0.71      1074
                  Positive emotion       0.57      0.62      0.59       594

                          accuracy                           0.63      1813
                         macro avg       0.42      0.48      0.44      1813
                      weighted avg       0.66      0.63      0.64      1813



## Evaluation